# ROGII - Wellbore Geology Prediction — submission notebook

Self-contained: no internet access needed (Code Competition requirement). Stage 2
baseline — per-well linear prior `tvt ~ MD + Z`, fit on each well's known rows and
predicted on that well's real evaluation zone (`TVT_input` is `NaN` there).

Locally verified (773 train wells): overall RMSE 67.09, median per-well 33.07 — far off
the ~4.86 public-LB leader. This is a **pipeline-proving baseline**, not a competitive
model — ships first so the submit path works end to end; see
`../context/05-plan-of-attack.md` for Stage 3+ (typewell/GR alignment, CNN) which is what
actually closes the gap.

`id = {WELLNAME}_{row_index}` where `row_index` is the 0-based row position in that
well's `__horizontal_well.csv` (confirmed against `sample_submission.csv`).

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Kaggle mounts competition data at /kaggle/input/<slug>/; fall back to the local
# dev copy (../data) so this notebook also runs outside Kaggle for testing.
_KAGGLE_DIR = "/kaggle/input/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def fit_predict_well(df):
    """Fit tvt ~ MD + Z on known rows, predict eval-zone rows.

    Returns (eval_positional_index, preds) where eval_positional_index is the
    0-based row position in df (matches the submission id's row_index).
    """
    df = df.reset_index(drop=True)
    known = df[df["TVT_input"].notna()]
    eval_rows = df[df["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return np.array([], dtype=int), np.array([])

    if len(known) >= 2:
        model = LinearRegression()
        model.fit(known[["MD", "Z"]], known["TVT_input"])
        preds = model.predict(eval_rows[["MD", "Z"]])
    else:
        # Degenerate well (too few known rows) - fall back to the known mean,
        # or 0.0 if there is no known signal at all.
        fallback = known["TVT_input"].mean() if len(known) else 0.0
        preds = np.full(len(eval_rows), fallback)

    return eval_rows.index.to_numpy(), preds

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"test wells: {len(test_wells)}")

rows = []
for well in test_wells:
    df = pd.read_csv(os.path.join(TEST_DIR, f"{well}__horizontal_well.csv"))
    eval_idx, preds = fit_predict_well(df)
    for idx, pred in zip(eval_idx, preds):
        rows.append({"id": f"{well}_{idx}", "tvt": pred})

submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(submission.shape)
submission.head()

In [ ]:
# Sanity checks against the sample submission before writing the real file.
sample = pd.read_csv(SAMPLE_SUBMISSION)

assert len(submission) == len(sample), (
    f"row count mismatch: {len(submission)} vs sample's {len(sample)}"
)
assert set(submission["id"]) == set(sample["id"]), "id sets don't match sample_submission"
assert submission["tvt"].notna().all(), "NaN predictions present - fix before submitting"

# Preserve sample_submission's row order so downstream tooling that assumes it
# doesn't break; scoring itself only cares about the id column, not row order.
submission = sample[["id"]].merge(submission, on="id", how="left")
assert submission["tvt"].notna().all()

print("All checks passed.")
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)